In [3]:
# LEVEL 2 — INVENTORY DATA CLEANING

import pandas as pd
from pathlib import Path

# Project directory
PROJECT_DIR = Path.cwd().parent

# Raw dataset directory
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"

# Work on a copy. Never modify the raw dataset.
inventory = pd.read_csv(RAW_DATA_DIR / "retail_inventory_ml_apl.csv")
inventory_clean = inventory.copy()

# 1. Standardize whitespace in text columns
text_columns = inventory_clean.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    inventory_clean[col] = inventory_clean[col].str.strip()

# 2. Convert date columns
inventory_clean["Start Date"] = pd.to_datetime(
    inventory_clean["Start Date"], errors="coerce"
)

inventory_clean["End Date"] = pd.to_datetime(
    inventory_clean["End Date"], errors="coerce"
)

# 3. Add data-quality flags
inventory_clean["flag_negative_stock"] = (
    inventory_clean["Qty on hand"] < 0
)

inventory_clean["flag_negative_unit_cost"] = (
    inventory_clean["Stock Unit Cost Price"] < 0
)

inventory_clean["flag_zero_selling_price"] = (
    inventory_clean["Stock Unit Selling Price"] == 0
)

inventory_clean["flag_zero_qty_residual_cost"] = (
    (inventory_clean["Qty on hand"] == 0) &
    (inventory_clean["Cost of Stocks"].abs() > 0.02)
)

# 4. Add calculated inventory metrics
inventory_clean["calculated_selling_value"] = (
    inventory_clean["Qty on hand"] *
    inventory_clean["Stock Unit Selling Price"]
)

inventory_clean["calculated_cost_value"] = (
    inventory_clean["Qty on hand"] *
    inventory_clean["Stock Unit Cost Price"]
)

# 5. Save the cleaned dataset
inventory_clean.to_csv(
    "../data/processed/inventory_clean.csv",
    index=False
)

print("INVENTORY CLEANING COMPLETE")
print("Original rows:", len(inventory))
print("Cleaned rows:", len(inventory_clean))
print("Columns after processing:", len(inventory_clean.columns))
print("Invalid start dates:", inventory_clean["Start Date"].isna().sum())
print("Invalid end dates:", inventory_clean["End Date"].isna().sum())
print("Negative stock flags:", inventory_clean["flag_negative_stock"].sum())
print("Negative cost flags:", inventory_clean["flag_negative_unit_cost"].sum())
print("Zero selling price flags:", inventory_clean["flag_zero_selling_price"].sum())
print("Zero quantity residual cost flags:", inventory_clean["flag_zero_qty_residual_cost"].sum())

print("\nSaved: data/processed/inventory_clean.csv")

INVENTORY CLEANING COMPLETE
Original rows: 284755
Cleaned rows: 284755
Columns after processing: 24
Invalid start dates: 0
Invalid end dates: 0
Negative stock flags: 1380
Negative cost flags: 25
Zero selling price flags: 13869
Zero quantity residual cost flags: 7542

Saved: data/processed/inventory_clean.csv


In [4]:
# LEVEL 2 — DEMAND DATA CLEANING

demand = pd.read_csv(RAW_DATA_DIR / "retail_store_inventory.csv")
demand_clean = demand.copy()

# Convert date
demand_clean["Date"] = pd.to_datetime(
    demand_clean["Date"],
    errors="coerce"
)

# Add data-quality flags
demand_clean["flag_negative_forecast"] = (
    demand_clean["Demand Forecast"] < 0
)

# Preserve original forecast; create a modelling-ready version
demand_clean["Demand Forecast Clean"] = (
    demand_clean["Demand Forecast"].clip(lower=0)
)

# Add inventory coverage metric
demand_clean["Inventory Coverage"] = (
    demand_clean["Inventory Level"] /
    demand_clean["Demand Forecast Clean"].replace(0, float("nan"))
)

# Save
demand_clean.to_csv(
    "../data/processed/demand_clean.csv",
    index=False
)

print("DEMAND CLEANING COMPLETE")
print("Rows:", len(demand_clean))
print("Invalid dates:", demand_clean["Date"].isna().sum())
print(
    "Negative forecast flags:",
    demand_clean["flag_negative_forecast"].sum()
)
print(
    "Negative forecasts retained:",
    (demand_clean["Demand Forecast"] < 0).sum()
)
print(
    "Clean forecast negatives:",
    (demand_clean["Demand Forecast Clean"] < 0).sum()
)

print("Saved: data/processed/demand_clean.csv")

DEMAND CLEANING COMPLETE
Rows: 73100
Invalid dates: 0
Negative forecast flags: 673
Negative forecasts retained: 673
Clean forecast negatives: 0
Saved: data/processed/demand_clean.csv


In [5]:
# LEVEL 2 — SALES DATA CLEANING

sales = pd.read_csv(RAW_DATA_DIR / "retail_sales_ml_apl.csv")
sales_clean = sales.copy()

# 1. Convert transaction date
sales_clean["Transaction Date"] = pd.to_datetime(
    sales_clean["Transaction Date"],
    errors="coerce"
)

# 2. Preserve return transactions
sales_clean["Is Return"] = sales_clean["Is Return"].astype("int8")

# 3. Add data-quality flags
sales_clean["flag_negative_quantity"] = (
    sales_clean["Qty Sold"] < 0
)

sales_clean["flag_negative_sales"] = (
    sales_clean["Sales Amount"] < 0
)

sales_clean["flag_negative_cogs"] = (
    sales_clean["Cogs"] < 0
)

# 4. Calculate net profit contribution
sales_clean["Net Profit Contribution"] = (
    sales_clean["Sales Amount"] - sales_clean["Cogs"]
)

# 5. Calculate return-aware sales metrics
sales_clean["Net Quantity"] = sales_clean["Qty Sold"]
sales_clean["Net Sales"] = sales_clean["Sales Amount"]
sales_clean["Net COGS"] = sales_clean["Cogs"]

# 6. Save
sales_clean.to_csv(
    "../data/processed/sales_clean.csv",
    index=False
)

print("SALES CLEANING COMPLETE")
print("Rows:", len(sales_clean))
print("Invalid dates:", sales_clean["Transaction Date"].isna().sum())
print("Return records:", sales_clean["Is Return"].sum())
print("Negative quantity flags:", sales_clean["flag_negative_quantity"].sum())
print("Negative sales flags:", sales_clean["flag_negative_sales"].sum())
print("Negative COGS flags:", sales_clean["flag_negative_cogs"].sum())
print("Net sales:", round(sales_clean["Net Sales"].sum(), 2))
print("Net profit contribution:", round(
    sales_clean["Net Profit Contribution"].sum(), 2
))
print("Saved: data/processed/sales_clean.csv")

SALES CLEANING COMPLETE
Rows: 125751
Invalid dates: 0
Return records: 7491
Negative quantity flags: 7491
Negative sales flags: 7491
Negative COGS flags: 7491
Net sales: 10482110.25
Net profit contribution: 4634359.15
Saved: data/processed/sales_clean.csv


In [6]:
# LEVEL 2 — FINAL VALIDATION

processed_files = {
    "Inventory": "../data/processed/inventory_clean.csv",
    "Demand": "../data/processed/demand_clean.csv",
    "Sales": "../data/processed/sales_clean.csv"
}

expected_rows = {
    "Inventory": 284755,
    "Demand": 73100,
    "Sales": 125751
}

validation_results = []

for name, path in processed_files.items():

    df = pd.read_csv(path)

    validation_results.append({
        "Dataset": name,
        "Expected Rows": expected_rows[name],
        "Actual Rows": len(df),
        "Row Count Passed": len(df) == expected_rows[name],
        "Duplicate Rows": df.duplicated().sum(),
        "File Exists": True
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print(
    "\nTOTAL PROCESSED RECORDS:",
    validation_df["Actual Rows"].sum()
)

print(
    "\nALL VALIDATIONS PASSED:",
    validation_df["Row Count Passed"].all()
    and validation_df["File Exists"].all()
)

,Dataset,Expected Rows,Actual Rows,Row Count Passed,Duplicate Rows,File Exists
0,Inventory,284755,284755,True,0,True
1,Demand,73100,73100,True,0,True
2,Sales,125751,125751,True,0,True



TOTAL PROCESSED RECORDS: 483606

ALL VALIDATIONS PASSED: True


In [8]:
from pathlib import Path
import pandas as pd

processed_files = {
    "Inventory": ("../data/processed/inventory_clean.csv", 284755),
    "Demand": ("../data/processed/demand_clean.csv", 73100),
    "Sales": ("../data/processed/sales_clean.csv", 125751)
}

results = []

for name, (path, expected) in processed_files.items():
    file_path = Path(path)
    df = pd.read_csv(file_path)

    results.append({
        "Dataset": name,
        "Expected Rows": expected,
        "Actual Rows": len(df),
        "Duplicates": int(df.duplicated().sum()),
        "Passed": len(df) == expected
    })

validation_df = pd.DataFrame(results)
display(validation_df)

print("\nTOTAL RECORDS:", validation_df["Actual Rows"].sum())
print("ALL VALIDATIONS PASSED:", validation_df["Passed"].all())

,Dataset,Expected Rows,Actual Rows,Duplicates,Passed
0,Inventory,284755,284755,0,True
1,Demand,73100,73100,0,True
2,Sales,125751,125751,0,True



TOTAL RECORDS: 483606
ALL VALIDATIONS PASSED: True
